# Aralin 09 - Disenyo ng Metacognition na Pattern


## Setup

Ipinapakita sa notebook na ito ang Metacognition design pattern gamit ang Microsoft Agent Framework.

**Mga Kinakailangan:**
- Azure OpenAI deployment na naka-configure gamit ang environment variables
- Azure CLI na authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Ano ang Metacognition?

Ang Metacognition ay **pag-iisip tungkol sa pag-iisip**. Sa konteksto ng mga AI na ahente, nangangahulugan ito ng paggawa ng mga ahenteng maaaring:

- **Magmuni-muni sa sarili** tungkol sa kanilang sariling mga output at proseso ng pangangatwiran
- **Makatuklas ng mga mali** at makabangon nang maayos sa halip na tahimik na bumigo
- **Suriin** kung ang kanilang mga tugon ay kumpleto at kapaki-pakinabang
- **Iangkop** ang kanilang estratehiya kapag ang paunang paraan ay hindi gumana (halimbawa, lumipat sa isang backup na sistema)

Ang isang metacognitive na ahente ay hindi lamang sumasagot ng mga tanong — ito ay nagmomonitor ng sariling pagganap at nag-aadjust nang mabilis.


## Pangunahing at Backup na Mga Kasangkapan

Isang karaniwang pattern sa metacognition ay ang **fallback strategy**. Sinusubukan muna ng ahente ang pangunahing kasangkapan; kung ito ay nabigo (hal., isang 404 error), kinikilala ng ahente ang pagkabigo at malinaw na lumilipat sa isang backup na kasangkapan.

Ito ay katulad ng mga totoong sistema kung saan maaaring hindi magamit ang mga pangunahing serbisyo at kailangang sariling suriin ng mga ahente ang problema bago pumili ng alternatibong landas.

Sa ibaba ay inilalarawan namin ang dalawang kasangkapan para sa paghahanap ng flight:
- **Pangunahing** — sumasaklaw sa Paris, Tokyo, at Barcelona
- **Backup** — sumasaklaw sa Berlin, Sydney, at New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Ahenteng Nagmumuni-muni sa Sarili na may Pagbawi sa Mali

Ang ahente sa ibaba ay inatasang subukan muna ang pangunahing sistema ng paglilipad, kilalanin ang mga pagkabigo, at malinaw na lumipat sa backup na sistema. Pagkatapos ng bawat sagot, maikling sinusuri nito sa sarili kung naipaliwanag nang buo ang tanong ng gumagamit.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Pattern ng Pagsusuri sa Sarili

Isa pang aspeto ng metacognition ay ang **pagsusuri sa sarili**: isang hiwalay na ahente (o ang parehong ahente sa ikalawang pasa) ang nire-review ang sagot para sa kabuuan, katumpakan, at kapakinabangan.

Sa ibaba ay gagawa tayo ng `ResponseEvaluator` na ahente na sumusukat sa mga sagot ng travel-agent sa tatlong dimensyon.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Buod

Sa araling ito natutunan mo kung paano gumawa ng **metacognitive agents** gamit ang Microsoft Agent Framework:

- **Pagmumuni-muni sa sarili**: Mga agent na nagbabantay sa sariling pangangatwiran at malinaw na nagsasabi kung ano ang nangyari.
- **Pagbangon mula sa pagkakamali gamit ang mga fallback**: Isang pattern na may pangunahing + backup na kasangkapan kung saan nalalaman ng agent ang mga pagkabigo (hal., 404 errors) at awtomatikong sumusubok ng alternatibong pinagmulan.
- **Pagsusuri sa sarili**: Isang hiwalay na evaluator agent na nagbibigay ng grado sa mga sagot para sa kasaklawan, katumpakan, at pagiging kapaki-pakinabang.

Ginagawa ng mga pattern na ito ang mga agent na mas matatag, malinaw, at mapagkakatiwalaan — mahalagang mga katangian para sa mga produktong deployment.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
